# Linear Algebra Preliminaries for Numerical Linear Algebra

This notebook develops the foundational concepts needed before studying NLA proper. Each section pairs mathematical statements with computation and visualization. The goal is not to prove everything rigorously but to build the geometric and computational intuition that makes algorithms like QR, SVD, and Krylov methods intelligible.

**Contents**
1. The Four Fundamental Subspaces
2. Linear Independence, Span, Basis, Dimension
3. Linear Maps and Matrix Representations
4. Inner Products and Norms
5. Orthogonal Projections
6. Eigenvalues and the Spectral Theorem
7. Rank and Near-Rank-Deficiency

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyArrowPatch, Polygon
from mpl_toolkits.mplot3d import Axes3D
from scipy import linalg
from scipy.linalg import null_space, orth

plt.rcParams.update({
    'figure.dpi': 120,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.size': 11,
    'axes.titlesize': 13,
    'axes.titleweight': 'bold'
})

rng = np.random.default_rng(42)
np.set_printoptions(precision=6, suppress=True)
print('Environment ready.')

---
## 1. The Four Fundamental Subspaces

Every matrix $A \in \mathbb{R}^{m \times n}$ induces four subspaces:

| Subspace | Lives in | Definition |
|---|---|---|
| Column space $\mathcal{C}(A)$ | $\mathbb{R}^m$ | $\{Ax : x \in \mathbb{R}^n\}$ — range of $A$ |
| Null space $\mathcal{N}(A)$ | $\mathbb{R}^n$ | $\{x : Ax = 0\}$ |
| Row space $\mathcal{C}(A^T)$ | $\mathbb{R}^n$ | Column space of $A^T$ |
| Left null space $\mathcal{N}(A^T)$ | $\mathbb{R}^m$ | $\{y : A^T y = 0\}$ |

**Fundamental Theorem of Linear Algebra (Part 1):**
$$\dim \mathcal{C}(A) = \dim \mathcal{C}(A^T) = r \quad\text{(rank of }A)$$
$$\dim \mathcal{N}(A) = n - r, \qquad \dim \mathcal{N}(A^T) = m - r$$

**Fundamental Theorem (Part 2) — the orthogonality relations:**
$$\mathcal{C}(A^T) \perp \mathcal{N}(A) \quad \text{in } \mathbb{R}^n$$
$$\mathcal{C}(A) \perp \mathcal{N}(A^T) \quad \text{in } \mathbb{R}^m$$

*Proof sketch:* If $Ax = 0$ and $v = A^T w$ is any row space vector, then $v \cdot x = w^T A x = w^T 0 = 0$. $\square$

In [ ]:
# Concrete example: A is 3x4 of rank 2
A = np.array([[1, 0,  1, 2],
              [0, 1,  1, 1],
              [1, 1,  2, 3]], dtype=float)

print(f'A ({A.shape[0]}x{A.shape[1]}):')
print(A)
print(f'\nnp.linalg.matrix_rank(A) = {np.linalg.matrix_rank(A)}')

# --- Column space basis via QR with column pivoting ---
Q, R, P = linalg.qr(A, pivoting=True)  # AP = QR
r = np.sum(np.abs(np.diag(R)) > 1e-10)
col_space_basis = Q[:, :r]
print(f'\nColumn space basis (cols of Q from QR):')
print(col_space_basis)

# --- Null space via SVD ---
U_svd, S, Vt = np.linalg.svd(A)
null_sp = Vt[r:, :].T  # rows of Vt corresponding to zero singular values
print(f'\nNull space basis (right singular vectors for sigma=0):')
print(null_sp)

# Verify: A @ null_sp should be ~0
print(f'\n||A @ null_space||_F = {np.linalg.norm(A @ null_sp):.2e}  (should be ~0)')

# Verify orthogonality: C(A^T) perp N(A)
row_space = Vt[:r, :].T
print(f'row_space^T @ null_space (should be ~0):')
print(np.round(row_space.T @ null_sp, 10))

# Rank-nullity
m, n = A.shape
print(f'\nRank-Nullity: rank(A) + nullity(A) = {r} + {null_sp.shape[1]} = {r + null_sp.shape[1]} = n = {n}')

In [ ]:
# Visualize: for a 2x2 rank-1 matrix, show the four subspaces as lines
A2 = np.array([[2, 1],
               [4, 2]], dtype=float)  # rank 1

_, _, Vt2 = np.linalg.svd(A2)
U2, _, _ = np.linalg.svd(A2)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, title, space1, space2, label1, label2, color1, color2 in [
    (axes[0], r'$\mathbb{R}^2$ (domain): Row space $\perp$ Null space',
     Vt2[0], Vt2[1], r'$\mathcal{C}(A^T)$ row space', r'$\mathcal{N}(A)$ null space',
     '#2E86AB', '#E84855'),
    (axes[1], r'$\mathbb{R}^2$ (codomain): Column space $\perp$ Left null space',
     U2[:, 0], U2[:, 1], r'$\mathcal{C}(A)$ col space', r'$\mathcal{N}(A^T)$ left null',
     '#44BBA4', '#F7A278')
]:
    t = np.linspace(-2, 2, 100)
    ax.plot(t * space1[0], t * space1[1], color=color1, lw=2.5, label=label1)
    ax.plot(t * space2[0], t * space2[1], color=color2, lw=2.5, ls='--', label=label2)
    ax.axhline(0, color='k', lw=0.5); ax.axvline(0, color='k', lw=0.5)
    ax.set_xlim(-2.5, 2.5); ax.set_ylim(-2.5, 2.5)
    ax.set_aspect('equal'); ax.legend(fontsize=9)
    ax.set_title(title)
    ax.plot([0.15, 0.15, 0.35], [0.0, 0.2, 0.2], color='gray', lw=1.5)  # right angle mark

plt.suptitle(r'Four Fundamental Subspaces of a $2\times 2$ rank-1 matrix', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()
print(f'Dot product of subspace directions: {space1 @ space2:.2e} (orthogonal)')

---
## 2. Linear Independence, Span, Basis, Dimension

**Definitions:**
- Vectors $v_1, \ldots, v_k$ are **linearly independent** if $\sum_i c_i v_i = 0 \implies c_i = 0\ \forall i$.
- Their **span** is $\{\sum_i c_i v_i : c_i \in \mathbb{R}\}$.
- A **basis** is a linearly independent spanning set for a subspace.
- All bases of a subspace have the same cardinality — that is the **dimension**.

**Computational test for independence:** The vectors $\{v_i\}$ are linearly dependent iff the matrix $V = [v_1 | \cdots | v_k]$ has $\det(V^T V) = 0$, equivalently the Gram matrix is singular. In NLA we detect near-dependence through small singular values.

**Why this matters for NLA:** Everything in NLA is ultimately about **choosing good bases** — QR finds an orthonormal basis for $\mathcal{C}(A)$, the SVD finds orthonormal bases for both the row space and column space simultaneously, Krylov methods build bases iteratively.

In [ ]:
# Three cases: independent, dependent, near-dependent
# Each V is 2x3: columns are the vectors being tested
cases = {
    'Independent': np.array([[1, 0, 0], [0, 1, 1]], dtype=float),      # v3 not in span(v1,v2)? Actually all 3 in R^2
    'Dependent': np.array([[1, 0, 1], [0, 1, 1]], dtype=float),         # v3 = v1 + v2
    'Near-dependent': np.array([[1, 0, 1+1e-8], [0, 1, 1+1e-8]], dtype=float)
}
# For 'Independent' use vectors that are actually independent as columns of a tall matrix
# Use 3D vectors instead so we can have genuinely independent vs dependent sets
cases = {
    'Independent': np.array([[1, 0, 0], [0, 1, 0], [0, 0, 1]], dtype=float).T,   # 3x3 identity cols -> 3x3
    'Dependent': np.array([[1, 0, 1], [0, 1, 1], [1, 1, 2]], dtype=float).T,     # v3 = v1+v2
    'Near-dependent': np.array([[1, 0, 1+1e-8], [0, 1, 1+1e-8], [0, 0, 1e-8]], dtype=float).T
}

print(f'{"Case":<20} {"Rank":<8} {"det(V^TV)":<18} {"Singular values"}')
print('-' * 65)
for name, V in cases.items():
    rank = np.linalg.matrix_rank(V)
    gram_det = np.linalg.det(V.T @ V)
    svs = np.linalg.svd(V, compute_uv=False)
    print(f'{name:<20} {rank:<8} {gram_det:<18.4e} {np.round(svs, 8)}')

In [ ]:
# Visualize span in R^2: independent vs dependent vectors
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left: two independent vectors span R^2
ax = axes[0]
ax.set_title("Two independent vectors span $\\mathbb{R}^2$", fontsize=13)
ax.set_xlim(-1, 4); ax.set_ylim(-1, 3.5)
ax.set_aspect('equal'); ax.grid(True, alpha=0.3)
u1, u2 = np.array([3, 1]), np.array([1, 2])
ax.annotate('', xy=u1, xytext=(0,0), arrowprops=dict(arrowstyle='->', color='#2E86AB', lw=2.5))
ax.annotate('', xy=u2, xytext=(0,0), arrowprops=dict(arrowstyle='->', color='#E84855', lw=2.5))
ax.text(u1[0]+0.1, u1[1]+0.1, '$v_1$', color='#2E86AB', fontsize=14)
ax.text(u2[0]+0.1, u2[1]+0.1, '$v_2$', color='#E84855', fontsize=14)
parallelogram = Polygon([np.zeros(2), u1, u1+u2, u2], alpha=0.15, color='purple')
ax.add_patch(parallelogram)
ax.text(1.8, 1.2, 'span = $\\mathbb{R}^2$', fontsize=12, color='purple')

# Right: two dependent vectors span only a line
ax = axes[1]
ax.set_title("Two dependent vectors span only a line", fontsize=13)
ax.set_xlim(-2, 4); ax.set_ylim(-1.5, 3)
ax.set_aspect('equal'); ax.grid(True, alpha=0.3)
w1, w2 = np.array([2, 1]), np.array([3, 1.5])
ax.annotate('', xy=w1, xytext=(0,0), arrowprops=dict(arrowstyle='->', color='#2E86AB', lw=2.5))
ax.annotate('', xy=w2, xytext=(0,0), arrowprops=dict(arrowstyle='->', color='#E84855', lw=2.5))
ax.text(w1[0]-0.3, w1[1]+0.2, '$v_1$', color='#2E86AB', fontsize=14)
ax.text(w2[0]+0.1, w2[1]+0.1, '$v_2 = 1.5 v_1$', color='#E84855', fontsize=14)
t = np.linspace(-1, 2, 50)
ax.plot(t*2, t*1, 'purple', lw=1.5, alpha=0.5)
ax.text(0.5, -0.8, 'span = a line (1D)', fontsize=12, color='purple')

plt.tight_layout()
plt.show()

---
## 3. Linear Maps and Matrix Representations

A map $T: \mathbb{R}^n \to \mathbb{R}^m$ is **linear** iff $T(\alpha u + \beta v) = \alpha T(u) + \beta T(v)$.

**Key fact:** Once we fix a basis $\{e_j\}$ for $\mathbb{R}^n$ and $\{f_i\}$ for $\mathbb{R}^m$, every linear map is represented by a unique matrix $A$ where $A_{ij} = [T(e_j)]_i$ (the $i$-th component of $T$ applied to the $j$-th basis vector).

**Consequence for NLA:** A change of basis from $\{e_j\}$ to $\{q_j\}$ transforms the matrix representation as:
$$A \longrightarrow B = Q^{-1} A Q$$
where $Q$ is the change-of-basis matrix. Eigendecomposition, SVD, and all factorizations are fundamentally about choosing bases that make $A$'s representation simple (diagonal, triangular, etc.).

In [ ]:
# Visualize a linear map as geometric transformation of the unit circle
theta = np.linspace(0, 2*np.pi, 300)
circle = np.vstack([np.cos(theta), np.sin(theta)])

transformations = {
    'Rotation $45°$': np.array([[np.cos(np.pi/4), -np.sin(np.pi/4)],
                                 [np.sin(np.pi/4),  np.cos(np.pi/4)]]),
    'Shear': np.array([[1, 1.5], [0, 1]]),
    'Projection onto $x$-axis': np.array([[1, 0], [0, 0]]),
    r'Stretch $\sigma_1=3, \sigma_2=0.5$': np.array([[3, 0], [0, 0.5]]),
}

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
colors = ['#2E86AB', '#44BBA4', '#E84855', '#F7A278']

for ax, (title, A_map), color in zip(axes, transformations.items(), colors):
    ax.plot(circle[0], circle[1], 'k--', lw=1, alpha=0.4, label='original')
    transformed = A_map @ circle
    ax.plot(transformed[0], transformed[1], color=color, lw=2, label='image')
    for e, ls in [(np.array([1, 0]), '-'), (np.array([0, 1]), '--')]:
        ax.annotate('', xy=e, xytext=(0,0), arrowprops=dict(arrowstyle='->', color='gray', lw=1.5, ls=ls))
        Ae = A_map @ e
        ax.annotate('', xy=Ae, xytext=(0,0), arrowprops=dict(arrowstyle='->', color=color, lw=2, ls=ls))
    ax.axhline(0, color='k', lw=0.4); ax.axvline(0, color='k', lw=0.4)
    lim = max(3.5, np.max(np.abs(transformed))*1.2)
    ax.set_xlim(-lim, lim); ax.set_ylim(-lim, lim)
    ax.set_aspect('equal'); ax.set_title(title, fontsize=10)
    ax.text(0.02, 0.98, f'det={np.linalg.det(A_map):.2f}', transform=ax.transAxes, va='top', fontsize=9, color='gray')

plt.suptitle('Linear maps acting on the unit circle — columns of $A$ are images of basis vectors', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Change of basis: same linear map, two different basis representations
A_cb = np.array([[3, 1], [0, 2]], dtype=float)

# New basis: orthonormal
Q_cb = np.array([[1, 1], [1, -1]], dtype=float) / np.sqrt(2)
B_cb = Q_cb.T @ A_cb @ Q_cb  # representation in new basis

print('Same linear map, two basis representations:')
print(f'Standard basis A:\n{A_cb}')
print(f'\nNew orthonormal basis Q:\n{np.round(Q_cb, 4)}')
print(f'\nNew basis B = Q^T A Q:\n{np.round(B_cb, 6)}')
print(f'\nEigenvalues of A: {np.sort(np.linalg.eigvals(A_cb)).real}')
print(f'Eigenvalues of B: {np.sort(np.linalg.eigvals(B_cb)).real}  (same — similarity invariant)')
print(f'Trace: A={np.trace(A_cb):.4f}, B={np.trace(B_cb):.4f}  (invariant)')
print(f'Det:   A={np.linalg.det(A_cb):.4f}, B={np.linalg.det(B_cb):.4f}  (invariant)')

---
## 4. Inner Products and Norms

The **standard inner product** on $\mathbb{R}^n$: $\langle x, y \rangle = x^T y = \sum_{i=1}^n x_i y_i$.

Two vectors are **orthogonal** if $\langle x, y \rangle = 0$.

### Vector Norms

A **norm** $\|\cdot\|: \mathbb{R}^n \to \mathbb{R}_{\ge 0}$ satisfies: (1) $\|x\| = 0 \iff x = 0$, (2) $\|\alpha x\| = |\alpha|\|x\|$, (3) $\|x+y\| \le \|x\| + \|y\|$.

| Name | Formula | Key property |
|---|---|---|
| 1-norm | $\|x\|_1 = \sum |x_i|$ | Sparsity, compressed sensing |
| 2-norm (Euclidean) | $\|x\|_2 = \sqrt{x^T x}$ | Orthogonality, geometry |
| $\infty$-norm | $\|x\|_\infty = \max |x_i|$ | Uniform convergence |
| $p$-norm | $\|x\|_p = (\sum |x_i|^p)^{1/p}$ | General family |

### Norm equivalence
In $\mathbb{R}^n$, all norms are equivalent: for any two norms $\|\cdot\|_\alpha, \|\cdot\|_\beta$ there exist constants $c_1, c_2 > 0$ such that $c_1 \|x\|_\alpha \leq \|x\|_\beta \leq c_2 \|x\|_\alpha$. The specific inequalities are:
$$\|x\|_\infty \leq \|x\|_2 \leq \|x\|_1, \qquad \|x\|_1 \leq \sqrt{n}\|x\|_2, \qquad \|x\|_2 \leq \sqrt{n}\|x\|_\infty$$

### Matrix Norms
- **Frobenius norm:** $\|A\|_F = \sqrt{\sum_{i,j} A_{ij}^2} = \sqrt{\sum_i \sigma_i^2}$
- **Induced (operator) 2-norm:** $\|A\|_2 = \max_{\|x\|_2=1} \|Ax\|_2 = \sigma_1(A)$ — the largest singular value
- **Condition number:** $\kappa(A) = \|A\|_2 \cdot \|A^{-1}\|_2 = \sigma_{\max}/\sigma_{\min}$

In [ ]:
# Visualize unit balls in different norms in R^2
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
ps = [1, 2, 4, np.inf]
colors = ['#2E86AB', '#44BBA4', '#E84855', '#F7A278']
labels = ['$p=1$ (diamond)', '$p=2$ (circle)', '$p=4$', r'$p=\infty$ (square)']

for ax, p, color, label in zip(axes, ps, colors, labels):
    if p == np.inf:
        x_vals = np.array([-1, 1, 1, -1, -1])
        y_vals = np.array([-1, -1, 1, 1, -1])
    else:
        x_half = np.linspace(-1, 1, 1000)
        y_half = (1 - np.abs(x_half)**p)**(1/p)
        x_vals = np.concatenate([x_half, x_half[::-1]])
        y_vals = np.concatenate([y_half, -y_half[::-1]])

    ax.fill(x_vals, y_vals, alpha=0.3, color=color)
    ax.plot(x_vals, y_vals, color=color, lw=2)
    ax.axhline(0, color='k', lw=0.4); ax.axvline(0, color='k', lw=0.4)
    ax.set_xlim(-1.5, 1.5); ax.set_ylim(-1.5, 1.5)
    ax.set_aspect('equal'); ax.set_title(label)

plt.suptitle(r'Unit balls $\{x \in \mathbb{R}^2 : \|x\|_p = 1\}$ for various $p$', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# --- Norm equivalence inequalities with actual numbers ---
x = np.array([3, -4, 0, 2, -1], dtype=float)
n_dim = len(x)
norm_1   = np.linalg.norm(x, 1)
norm_2   = np.linalg.norm(x, 2)
norm_inf = np.linalg.norm(x, np.inf)

print(f"x = {x},  n = {n_dim}")
print(f"  ||x||_1   = {norm_1:.4f}")
print(f"  ||x||_2   = {norm_2:.4f}")
print(f"  ||x||_inf = {norm_inf:.4f}")
print(f"\nNorm equivalence inequalities (verified numerically):")
print(f"  ||x||_inf <= ||x||_2 <= ||x||_1:         {norm_inf:.2f} <= {norm_2:.2f} <= {norm_1:.2f}")
print(f"  ||x||_1   <= sqrt(n)*||x||_2:             {norm_1:.2f} <= {np.sqrt(n_dim)*norm_2:.2f}")
print(f"  ||x||_2   <= sqrt(n)*||x||_inf:           {norm_2:.2f} <= {np.sqrt(n_dim)*norm_inf:.2f}")

# --- Induced 2-norm = largest singular value ---
A_norm = np.array([[3, 1], [1, 2]], dtype=float)
U_n, S_n, Vt_n = np.linalg.svd(A_norm)

# Verify by Monte Carlo sampling on unit sphere
N_samp = 100000
x_rand = rng.standard_normal((2, N_samp))
x_rand /= np.linalg.norm(x_rand, axis=0)
norms_Ax = np.linalg.norm(A_norm @ x_rand, axis=0)

print(f'\n--- Matrix norms for A = [[3,1],[1,2]] ---')
print(f'Singular values: s1={S_n[0]:.6f}, s2={S_n[1]:.6f}')
print(f'||A||_2 = s1 = {S_n[0]:.6f}')
print(f'Max ||Ax||_2 over {N_samp} unit vectors = {norms_Ax.max():.6f}  (matches s1)')
print(f'||A||_F = sqrt(s1^2+s2^2) = {np.sqrt(S_n[0]**2+S_n[1]**2):.6f}')
print(f'kappa_2(A) = s1/s2 = {S_n[0]/S_n[1]:.6f}')

In [ ]:
# A maps unit circle to ellipse. Semi-axes are sigma_i * u_i.
A_svd = np.array([[3, 1], [1, 2]], dtype=float)
U_s, S_s, Vt_s = np.linalg.svd(A_svd)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
theta_vals = np.linspace(0, 2*np.pi, 300)
unit_circ = np.vstack([np.cos(theta_vals), np.sin(theta_vals)])
ellipse = A_svd @ unit_circ

# Left: input space with right singular vectors
axes[0].plot(unit_circ[0], unit_circ[1], 'k-', lw=1.5, label='unit circle')
for i, (v, color) in enumerate(zip(Vt_s, ['#2E86AB', '#E84855'])):
    axes[0].annotate('', xy=v, xytext=(0,0), arrowprops=dict(arrowstyle='->', color=color, lw=2.5))
    axes[0].text(v[0]+0.05, v[1]+0.05, f'$v_{i+1}$', color=color, fontsize=14)
axes[0].set_xlim(-1.5, 1.5); axes[0].set_ylim(-1.5, 1.5)
axes[0].set_aspect('equal'); axes[0].set_title('Input: right singular vectors $v_i$')
axes[0].axhline(0, color='k', lw=0.4); axes[0].axvline(0, color='k', lw=0.4)

# Right: output space with sigma_i * u_i
axes[1].plot(ellipse[0], ellipse[1], color='#44BBA4', lw=2, label='image ellipse')
for i, (u, s, color) in enumerate(zip(U_s.T, S_s, ['#2E86AB', '#E84855'])):
    axes[1].annotate('', xy=s*u, xytext=(0,0), arrowprops=dict(arrowstyle='->', color=color, lw=2.5))
    axes[1].text(s*u[0]+0.05, s*u[1]+0.05, rf'$\sigma_{i+1} u_{i+1}$', color=color, fontsize=12)
axes[1].set_xlim(-4.5, 4.5); axes[1].set_ylim(-4.5, 4.5)
axes[1].set_aspect('equal')
axes[1].set_title(rf'Output: image ellipse, semi-axes $\sigma_1={S_s[0]:.2f}, \sigma_2={S_s[1]:.2f}$')
axes[1].axhline(0, color='k', lw=0.4); axes[1].axvline(0, color='k', lw=0.4)

plt.suptitle(r'$A$ maps unit circle $\to$ ellipse. $\|A\|_2 = \sigma_1 =$ length of longest semi-axis.', fontweight='bold')
plt.tight_layout()
plt.show()

---
## 5. Orthogonal Projections

Let $S$ be a subspace of $\mathbb{R}^m$ with orthonormal basis $Q = [q_1 | \cdots | q_r]$. The **orthogonal projector** onto $S$ is:
$$P = Q Q^T$$

**Properties:** $P^2 = P$ (idempotent), $P^T = P$ (symmetric), $I - P$ is the projector onto $S^\perp$.

**Geometric meaning:** For any $b \in \mathbb{R}^m$, $Pb$ is the unique vector in $S$ closest to $b$:
$$Pb = \arg\min_{v \in S} \|b - v\|_2$$

**Proof:** Write $b = Pb + (b - Pb)$. For any $v \in S$:
$$\|b - v\|^2 = \|(Pb - v) + (b - Pb)\|^2 = \|Pb - v\|^2 + \|b - Pb\|^2$$
since $Pb - v \in S$ and $b - Pb \in S^\perp$ are orthogonal. This is minimized when $v = Pb$. $\square$

This is the geometric engine behind **least squares**: $\hat{x} = (A^T A)^{-1} A^T b$ gives $A\hat{x} = Pb$ where $P$ projects onto $\mathcal{C}(A)$.

In [ ]:
# Verify projector properties algebraically
Q_raw = np.array([[1, 0], [1, 1], [0, 1]], dtype=float)
Q_proj, _ = np.linalg.qr(Q_raw)
Q_proj = Q_proj[:, :2]  # orthonormal basis for a plane in R^3

P = Q_proj @ Q_proj.T  # projector

print('Projector P = QQ^T:')
print(np.round(P, 6))
print(f'\nP^2 == P? max|P^2 - P| = {np.max(np.abs(P @ P - P)):.2e}')
print(f'P^T == P? max|P^T - P| = {np.max(np.abs(P.T - P)):.2e}')
print(f'Eigenvalues of P: {np.round(np.sort(np.linalg.eigvals(P).real)[::-1], 4)}')
print('  (Should be: r ones, m-r zeros. r=2, m=3 -> [1, 1, 0])')

# Projection minimizes distance
b = np.array([3, 1, 2], dtype=float)
Pb = P @ b
print(f'\nb = {b}')
print(f'Pb = {np.round(Pb, 4)}  (projection onto plane)')
print(f'Residual r = b - Pb = {np.round(b - Pb, 4)}')
print(f'r perp C(Q)? Q^T r = {np.round(Q_proj.T @ (b - Pb), 10)}  (should be 0)')

In [ ]:
# Visualize projection in 3D
fig = plt.figure(figsize=(10, 7))
ax = fig.add_subplot(111, projection='3d')

# Draw the plane
xx, yy = np.meshgrid(np.linspace(-1, 4, 20), np.linspace(-1, 4, 20))
_, _, Vt_plane = np.linalg.svd(Q_proj.T)
n_vec = Vt_plane[-1]  # normal to the plane
zz = (-n_vec[0] * xx - n_vec[1] * yy) / n_vec[2]
ax.plot_surface(xx, yy, zz, alpha=0.15, color='#2E86AB')

# b, Pb, residual
ax.quiver(0, 0, 0, *b, color='#E84855', lw=2, arrow_length_ratio=0.1, label=r'$b$')
ax.quiver(0, 0, 0, *Pb, color='#2E86AB', lw=2.5, arrow_length_ratio=0.1, label=r'$Pb$ (projection)')
ax.quiver(*Pb, *(b-Pb), color='#44BBA4', lw=2, arrow_length_ratio=0.15, label=r'$b - Pb$ (residual, $\perp$ plane)')

ax.set_xlabel('x'); ax.set_ylabel('y'); ax.set_zlabel('z')
ax.set_title('Orthogonal Projection: $Pb$ is the closest point in the subspace to $b$', fontweight='bold')
ax.legend(fontsize=10)
ax.set_xlim(0, 3.5); ax.set_ylim(0, 2); ax.set_zlim(0, 3)
plt.tight_layout()
plt.show()

In [ ]:
# Least squares as projection: overdetermined system Ax = b
# Fit a quadratic to noisy data
t = np.linspace(0, 1, 20)
b_noisy = 2 + 3*t - 5*t**2 + 0.3 * rng.standard_normal(20)

# Vandermonde matrix: columns are 1, t, t^2
A_ls = np.column_stack([np.ones_like(t), t, t**2])
x_ls, residuals, rank_ls, sv_ls = np.linalg.lstsq(A_ls, b_noisy, rcond=None)
Pb_ls = A_ls @ x_ls
r_ls = b_noisy - Pb_ls

print(f'Least squares solution: x = {np.round(x_ls, 4)}')
print(f'True coefficients:          [2.0000, 3.0000, -5.0000]')
print(f'\nA^T r (should be ~0): {np.round(A_ls.T @ r_ls, 10)}')
print(f'||r||_2 = {np.linalg.norm(r_ls):.6f}  (minimum possible residual norm)')

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
t_fine = np.linspace(0, 1, 200)
A_fine = np.column_stack([np.ones_like(t_fine), t_fine, t_fine**2])

axes[0].scatter(t, b_noisy, color='#E84855', zorder=5, label='data $b$')
axes[0].plot(t_fine, A_fine @ x_ls, color='#2E86AB', lw=2, label=r'fit $A\hat{x} = Pb$')
axes[0].plot(t_fine, 2 + 3*t_fine - 5*t_fine**2, 'k--', lw=1.5, alpha=0.5, label='true curve')
for ti, bi, pbi in zip(t, b_noisy, Pb_ls):
    axes[0].plot([ti, ti], [bi, pbi], color='#44BBA4', lw=0.8, alpha=0.7)
axes[0].set_title('Least Squares = Projection onto $\\mathcal{C}(A)$'); axes[0].legend()

axes[1].bar(range(3), np.abs(A_ls.T @ r_ls), color=['#2E86AB', '#44BBA4', '#E84855'])
axes[1].set_xticks(range(3)); axes[1].set_xticklabels(['$a_1^T r$', '$a_2^T r$', '$a_3^T r$'])
axes[1].set_ylabel('|value|'); axes[1].set_yscale('log')
axes[1].set_title('Residual $r \\perp$ all columns of $A$\n(Normal equations satisfied)')

plt.tight_layout()
plt.show()

---
## 6. Eigenvalues and the Spectral Theorem

A scalar $\lambda$ and nonzero vector $v$ satisfy $Av = \lambda v$ — $v$ is an **eigenvector**, $\lambda$ the **eigenvalue**.

**Key facts:** $\text{tr}(A) = \sum_i \lambda_i$, $\det(A) = \prod_i \lambda_i$, similar matrices share eigenvalues.

### The Spectral Theorem
If $A \in \mathbb{R}^{n \times n}$ is symmetric ($A = A^T$), then:
1. All eigenvalues are **real**.
2. Eigenvectors for distinct eigenvalues are **orthogonal**.
3. $A$ has an orthonormal eigenbasis: $A = Q \Lambda Q^T$ where $Q$ is orthogonal, $\Lambda = \text{diag}(\lambda_1, \ldots, \lambda_n)$.

**Rayleigh quotient:** $R(A, x) = \frac{x^T A x}{x^T x}$ satisfies $\lambda_{\min} \leq R(A,x) \leq \lambda_{\max}$, with equality at the corresponding eigenvectors. This variational principle underlies power iteration and Krylov methods.

**Why symmetric matrices dominate NLA:** $A^T A$, $AA^T$, covariance matrices, stiffness matrices, Hessians — all symmetric. The spectral theorem guarantees clean diagonalization: no Jordan blocks, no complex arithmetic.

In [ ]:
# --- Non-symmetric vs Symmetric: the critical contrast ---
# Non-symmetric: eigenvalues can be complex
A_nonsym = np.array([[0, -1],
                     [1,  0]], dtype=float)  # 90° rotation
evals_ns = np.linalg.eigvals(A_nonsym)
print("Non-symmetric matrix (90° rotation):")
print(A_nonsym)
print(f"  Eigenvalues: {evals_ns}  (complex! No real eigenvectors in R^2)")
print()

# Symmetric: real eigenvalues, orthogonal eigenvectors (spectral theorem)
rng2 = np.random.default_rng(7)
M_raw = rng2.standard_normal((4, 4))
A_sym = M_raw @ M_raw.T  # symmetric positive definite
eigvals_sym, eigvecs_sym = np.linalg.eigh(A_sym)
eigvals_sym = eigvals_sym[::-1]; eigvecs_sym = eigvecs_sym[:, ::-1]

print('Symmetric matrix A = M M^T:')
print(np.round(A_sym, 4))
print(f'\nEigenvalues (all real): {np.round(eigvals_sym, 6)}')
print(f'\nQ^T Q (should be I):')
print(np.round(eigvecs_sym.T @ eigvecs_sym, 10))
print(f'\n||A - Q Lambda Q^T||_F = {np.linalg.norm(A_sym - eigvecs_sym @ np.diag(eigvals_sym) @ eigvecs_sym.T):.2e}')

# Rayleigh quotient bounds
x_test = rng2.standard_normal((4, 10000))
x_test /= np.linalg.norm(x_test, axis=0)
RQ = np.einsum('ij,ij->j', x_test, A_sym @ x_test)
print(f'\nRayleigh quotient bounds:')
print(f'  lam_min = {eigvals_sym[-1]:.6f},  min R(A,x) = {RQ.min():.6f}')
print(f'  lam_max = {eigvals_sym[0]:.6f},  max R(A,x) = {RQ.max():.6f}')

In [ ]:
# Visualize: eigenvectors as principal axes + Rayleigh quotient over unit circle
A_2d = np.array([[3, 1.5], [1.5, 1.5]])
eigvals_2d, eigvecs_2d = np.linalg.eigh(A_2d)
eigvals_2d = eigvals_2d[::-1]; eigvecs_2d = eigvecs_2d[:, ::-1]

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Left: quadratic form level sets
x_grid = np.linspace(-2, 2, 400)
X, Y = np.meshgrid(x_grid, x_grid)
Z = A_2d[0,0]*X**2 + 2*A_2d[0,1]*X*Y + A_2d[1,1]*Y**2
c = axes[0].contourf(X, Y, Z, levels=20, cmap='RdYlBu_r', alpha=0.8)
plt.colorbar(c, ax=axes[0])
axes[0].contour(X, Y, Z, levels=[0.5, 1, 2, 4], colors='white', alpha=0.6, linewidths=1)

for i, (ev, lam, color) in enumerate(zip(eigvecs_2d.T, eigvals_2d, ['#2E86AB', '#E84855'])):
    scale = 1.0 / np.sqrt(lam)
    axes[0].annotate('', xy=scale*ev, xytext=(0,0), arrowprops=dict(arrowstyle='->', color=color, lw=3))
    axes[0].annotate('', xy=-scale*ev, xytext=(0,0), arrowprops=dict(arrowstyle='->', color=color, lw=3))
    axes[0].text(scale*ev[0]+0.05, scale*ev[1]+0.05, rf'$q_{i+1}$, $\lambda_{i+1}={lam:.2f}$', color=color, fontsize=11)
axes[0].set_xlim(-2, 2); axes[0].set_ylim(-2, 2); axes[0].set_aspect('equal')
axes[0].set_title('Level sets of $x^T A x$\nEigenvectors = principal axes')
axes[0].axhline(0, color='w', lw=0.5); axes[0].axvline(0, color='w', lw=0.5)

# Right: Rayleigh quotient as function of angle
theta_rq = np.linspace(0, 2*np.pi, 500)
x_unit = np.vstack([np.cos(theta_rq), np.sin(theta_rq)])
rq_vals = np.einsum('ij,ij->j', x_unit, A_2d @ x_unit)
axes[1].plot(np.degrees(theta_rq), rq_vals, color='#2E86AB', lw=2)
axes[1].axhline(eigvals_2d[0], color='#E84855', ls='--', lw=1.5, label=rf'$\lambda_1 = {eigvals_2d[0]:.3f}$')
axes[1].axhline(eigvals_2d[1], color='#44BBA4', ls='--', lw=1.5, label=rf'$\lambda_2 = {eigvals_2d[1]:.3f}$')
axes[1].set_xlabel('angle (degrees)'); axes[1].set_ylabel('$R(A, x(\\theta))$')
axes[1].set_title('Rayleigh quotient over unit circle\nBounded by $[\\lambda_{\\min}, \\lambda_{\\max}]$')
axes[1].legend()
for i, ev in enumerate(eigvecs_2d.T):
    ang = np.degrees(np.arctan2(ev[1], ev[0])) % 360
    axes[1].axvline(ang, color='gray', ls=':', lw=1)

plt.suptitle('Spectral Theorem: eigenvectors = principal axes of quadratic form', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Spectral decomposition visualized as sum of rank-1 matrices
A2_sym = np.array([[3, 1], [1, 2]], dtype=float)
evals2, evecs2 = np.linalg.eigh(A2_sym)

# A = lam1*v1*v1^T + lam2*v2*v2^T
fig, ax = plt.subplots(figsize=(10, 3))

rank1_components = []
for i in range(2):
    v = evecs2[:, i:i+1]
    rank1_components.append(evals2[i] * (v @ v.T))

labels = ['$A$', '=', rf'$\lambda_1 v_1 v_1^T$', '+', rf'$\lambda_2 v_2 v_2^T$']
matrices = [A2_sym, None, rank1_components[0], None, rank1_components[1]]

x_pos = 0
for label, mat in zip(labels, matrices):
    if mat is not None:
        ax.text(x_pos, 0.5, f'[{mat[0,0]:+.3f}  {mat[0,1]:+.3f}]\n[{mat[1,0]:+.3f}  {mat[1,1]:+.3f}]',
                fontsize=12, family='monospace', ha='center', va='center',
                bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))
        ax.text(x_pos, 1.1, label, fontsize=14, ha='center', va='center', fontweight='bold')
    else:
        ax.text(x_pos, 0.5, label, fontsize=20, ha='center', va='center')
    x_pos += 2.0

ax.text(x_pos + 0.5, 0.5, rf'$\lambda_1={evals2[0]:.3f},\ \lambda_2={evals2[1]:.3f}$',
        fontsize=12, ha='left', va='center', style='italic', color='gray')

ax.set_xlim(-1.5, x_pos + 3); ax.set_ylim(-0.3, 1.5)
ax.axis('off')
ax.set_title('Spectral decomposition: $A = \\sum_i \\lambda_i v_i v_i^T$  (sum of rank-1 symmetric matrices)',
             fontweight='bold', fontsize=12)
plt.tight_layout()
plt.show()

---
## 7. Rank and Near-Rank-Deficiency

The **rank** of $A$ is $r = \dim \mathcal{C}(A) = \dim \mathcal{C}(A^T)$, equivalently the number of nonzero singular values.

**Rank deficiency** ($r < \min(m,n)$) has two distinct effects:
- Theoretical: $Ax = b$ has either no solution or infinitely many.
- Numerical: *near* rank deficiency — small but nonzero singular values — leads to catastrophic amplification of errors. This is why we care about **condition number** $\kappa(A) = \sigma_1 / \sigma_r$.

**The rank-revealing SVD:** $A = U \Sigma V^T$ simultaneously reveals:
- The rank (number of nonzero $\sigma_i$)
- An orthonormal basis for each of the four fundamental subspaces
- The condition number $\kappa = \sigma_1 / \sigma_r$
- The best rank-$k$ approximation (Eckart-Young theorem)

In [ ]:
# The Hilbert matrix: the classic example of near-rank deficiency
def hilbert(n):
    return np.array([[1.0/(i+j+1) for j in range(n)] for i in range(n)])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: singular value decay for Hilbert matrices of increasing size
ax = axes[0]
for n in [4, 6, 8, 10, 12]:
    H = hilbert(n)
    sigmas = np.linalg.svd(H, compute_uv=False)
    ax.semilogy(range(1, n+1), sigmas, 'o-', label=f'n={n}', markersize=4)
ax.set_xlabel('Singular value index')
ax.set_ylabel('$\\sigma_i$ (log scale)')
ax.set_title('Singular value decay of Hilbert matrices')
ax.legend(fontsize=10); ax.grid(True, alpha=0.3)

# Right: condition number growth
ax = axes[1]
sizes = range(2, 16)
conds = []
for n in sizes:
    conds.append(np.linalg.cond(hilbert(n)))
ax.semilogy(list(sizes), conds, 'ro-', markersize=6)
ax.set_xlabel('Matrix size $n$')
ax.set_ylabel('$\\kappa_2(H_n)$ (log scale)')
ax.set_title('Condition number of Hilbert matrices')
ax.axhline(1/np.finfo(float).eps, color='gray', ls='--',
           label=rf'$1/\epsilon_{{mach}} \approx {1/np.finfo(float).eps:.0e}$')
ax.legend(fontsize=10); ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

eps_mach = np.finfo(float).eps
print(f"Machine epsilon: {eps_mach:.2e}")
for n in [5, 8, 12]:
    kappa = np.linalg.cond(hilbert(n))
    digits = max(0, int(np.log10(1/eps_mach)) - int(np.log10(kappa)))
    print(f"  H_{n}: kappa_2 ~ {kappa:.2e}, expected accurate digits ~ {digits}")

In [ ]:
# Sensitivity of linear systems to near-rank deficiency
np.random.seed(42)

print(f'{"System":<22} {"kappa(H)":<14} {"||db||/||b||":<14} {"||dx||/||x||":<14} {"Amplification":<14} {"Bound (kappa)"}')
print('-' * 80)
for n, label in [(5, "H_5  (moderate kappa)"), (8, "H_8  (large kappa)"), (12, "H_12 (extreme kappa)")]:
    H = hilbert(n)
    x_true = np.ones(n)
    b = H @ x_true

    # Perturb b slightly
    delta_b = 1e-10 * np.random.randn(n)
    x_pert = np.linalg.solve(H, b + delta_b)

    rel_err_b = np.linalg.norm(delta_b) / np.linalg.norm(b)
    rel_err_x = np.linalg.norm(x_pert - x_true) / np.linalg.norm(x_true)
    kappa = np.linalg.cond(H)
    amplification = rel_err_x / rel_err_b

    print(f'{label:<22} {kappa:<14.2e} {rel_err_b:<14.2e} {rel_err_x:<14.2e} {amplification:<14.2e} {kappa:.2e}')

print(f'\nKey insight: ||dx||/||x|| <= kappa(A) * ||db||/||b||')
print(f'When kappa ~ 1/eps_mach, the computed solution may have NO correct digits.')

In [ ]:
# Eckart-Young theorem: SVD gives best low-rank approximation
# ||A - A_k||_2 = s_{k+1}  and  ||A - A_k||_F = sqrt(s_{k+1}^2 + ... + s_r^2)

rng3 = np.random.default_rng(13)
m_ey, n_ey = 80, 80
r_true = 5
L = rng3.standard_normal((m_ey, r_true)) @ rng3.standard_normal((r_true, n_ey))  # rank-5 signal
noise = 0.5 * rng3.standard_normal((m_ey, n_ey))
A_noisy = L + noise

U_ey, S_ey, Vt_ey = np.linalg.svd(A_noisy)

ks = range(1, 20)
errors_2 = [S_ey[k] for k in ks]
errors_F = [np.sqrt(np.sum(S_ey[k:]**2)) for k in ks]

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Singular value spectrum
axes[0].semilogy(range(1, len(S_ey)+1), S_ey, 'o-', color='#2E86AB', ms=4)
axes[0].axvline(r_true, color='#E84855', ls='--', lw=2, label=f'true rank = {r_true}')
axes[0].set_xlabel('index $i$'); axes[0].set_ylabel('$\\sigma_i$')
axes[0].set_title('Singular Value Spectrum\n(gap reveals true rank)'); axes[0].legend()

# Approximation errors
axes[1].semilogy(list(ks), errors_2, 's-', color='#44BBA4', label=r'$\|A - A_k\|_2 = \sigma_{k+1}$')
axes[1].semilogy(list(ks), errors_F, '^-', color='#E84855', label=r'$\|A - A_k\|_F$')
axes[1].axvline(r_true, color='gray', ls=':', lw=2, label=f'true rank = {r_true}')
axes[1].set_xlabel('rank $k$'); axes[1].set_ylabel('approximation error')
axes[1].set_title('Eckart-Young: Best Rank-$k$ Error'); axes[1].legend(fontsize=9)

# Rank-5 reconstruction
A_k = U_ey[:, :r_true] @ np.diag(S_ey[:r_true]) @ Vt_ey[:r_true, :]
vmin, vmax = A_noisy.min(), A_noisy.max()
axes[2].imshow(A_k, cmap='viridis', vmin=vmin, vmax=vmax, aspect='auto')
axes[2].set_title(f'Rank-{r_true} approximation\n(captures signal, removes noise)')
axes[2].set_xlabel('col'); axes[2].set_ylabel('row')

plt.suptitle('Eckart-Young Theorem: SVD gives optimal low-rank approximation', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# The SVD unifies all seven preliminaries
print('=' * 60)
print('SVD UNIFIES ALL SEVEN PRELIMINARIES')
print('=' * 60)

A_summary = np.array([[1, 2, 0],
                       [0, 1, 1],
                       [1, 3, 1],
                       [2, 1, -1]], dtype=float)

U_sum, S_sum, Vt_sum = np.linalg.svd(A_summary, full_matrices=True)
r_sum = np.sum(S_sum > 1e-10)

print(f'\nA: {A_summary.shape[0]}x{A_summary.shape[1]}, rank = {r_sum}')
print(f'\n1. FOUR SUBSPACES:')
print(f'   C(A):   dim={r_sum}, basis = U[:,:{r_sum}]')
print(f'   N(A^T): dim={A_summary.shape[0]-r_sum}, basis = U[:,{r_sum}:]')
print(f'   C(A^T): dim={r_sum}, basis = Vt[:{r_sum},:].T')
print(f'   N(A):   dim={A_summary.shape[1]-r_sum}, basis = Vt[{r_sum}:,:].T')
print(f'\n2. BASIS/DIMENSION: column space dim = row space dim = {r_sum}')
print(f'\n3. LINEAR MAP: A = U S V^T decomposes into rotate x stretch x rotate')
print(f'\n4. NORMS: ||A||_2 = s_1 = {S_sum[0]:.4f},  ||A||_F = {np.linalg.norm(A_summary, "fro"):.4f} = sqrt(sum s_i^2)')
P_col = U_sum[:, :r_sum] @ U_sum[:, :r_sum].T
b_test = np.array([1, 0, 0, 1], dtype=float)
print(f'\n5. PROJECTION: P = UU^T projects onto C(A)')
print(f'   ||Pb - b||_2 = {np.linalg.norm(P_col @ b_test - b_test):.4f}')
print(f'\n6. EIGENVALUES: A^T A has eigenvalues = s_i^2 = {np.round(S_sum**2, 4)}')
print(f'   eigvecs of A^T A = rows of V^T')
print(f'\n7. RANK/CONDITION: kappa(A) = s_1/s_r = {S_sum[0]/S_sum[r_sum-1]:.4f}')

---
## Summary

The seven preliminaries form a coherent structure, not a list:

```
Vector spaces + linear maps
       ↓
Span, independence, basis, dimension
       ↓
Inner products → orthogonality → norms
       ↓
Orthogonal projections  ←→  Least squares
       ↓
Eigenvalues/Spectral Theorem  ←→  Symmetric systems
       ↓
Rank, singular values, condition number
       ↓
      SVD  ←— unifies all of the above
```

From here, Trefethen & Bau Chapter 1-3 (QR/Gram-Schmidt), Chapter 4-5 (least squares), and Chapter 6+ (eigenvalue algorithms) all build directly on this foundation.